# RAG PDF Research Corpus System - Quick Start Example

**Version:** 1.0.0  
**Date:** 2025-11-25  
**Purpose:** Pre-configured example for getting started quickly

---

## About This Notebook

This is a streamlined, pre-configured example that demonstrates the core functionality
of the RAG PDF Research Corpus System. It includes:

- ✅ Pre-populated configuration
- ✅ Step-by-step execution cells
- ✅ Expected output examples
- ✅ Common use cases

---

## Step 1: Environment Setup

First, install all required dependencies.

In [ ]:
# Install dependencies (run once)
!pip install -q openai>=1.3.0 langgraph>=0.0.30 langchain>=0.1.0
!pip install -q pymupdf>=1.23.0 faiss-cpu>=1.7.4 scikit-learn>=1.3.0
!pip install -q pydantic>=2.0.0 pandas>=2.0.0 numpy>=1.24.0
!pip install -q tqdm>=4.65.0 matplotlib>=3.7.0 seaborn>=0.12.0
!pip install -q requests>=2.31.0 python-dateutil>=2.8.2

print("✅ Dependencies installed successfully!")

In [ ]:
# Verify Python version
import sys
print(f"Python version: {sys.version}")

if sys.version_info >= (3, 10):
    print("✅ Python 3.10+ requirement met")
else:
    print("⚠️ Python 3.10+ recommended. Some features may not work.")

## Step 2: Set Up API Key

You'll need an OpenAI API key. Get one at: https://platform.openai.com/api-keys

In [ ]:
import os
from getpass import getpass

# Securely input your API key
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
    print("✅ API key set")
else:
    print("✅ API key already set")

## Step 3: Mount Google Drive

Mount Google Drive to access your PDF files.

In [ ]:
# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("✅ Google Drive mounted successfully")
except ImportError:
    IN_COLAB = False
    print("ℹ️ Not running in Colab. Using local file system.")

## Step 4: Import Modules

Import the RAG PDF system modules.

In [ ]:
# Add repository to path (if needed)
import sys
sys.path.insert(0, '/content/drive/MyDrive/research_corpus_organizer')  # Adjust path as needed

# Import core modules
from rag_models import (
    RunConfig, 
    create_default_config, 
    PaperRecord, 
    StateManager
)
from workflow_orchestrator import (
    run_full_pipeline,
    run_ingestion_only,
    save_checkpoint,
    load_checkpoint,
    display_workflow_state,
    print_cost_summary,
)
from rag_query_interface import RAGQueryEngine, interactive_query
from quality_control import create_qc_dashboard, generate_qc_report

print("✅ All modules imported successfully")

## Step 5: Configure the Pipeline

Create a configuration for your corpus. Adjust these settings based on your needs.

In [ ]:
# =============================================================================
# PRE-CONFIGURED SETTINGS - Adjust as needed
# =============================================================================

config = create_default_config(
    # ===== YOUR SETTINGS =====
    # Change this to your PDF folder path in Google Drive
    drive_folder_path="Research_PDFs",
    
    # ===== PROCESSING LIMITS =====
    max_papers_per_run=20,        # Start small, increase later
    max_chunks_per_paper=50,      # Chunks per paper
    
    # ===== MODELS =====
    summary_model="gpt-5-mini",   # Cost-effective model
    taxonomy_model="gpt-5-mini",
    classification_model="gpt-5-mini",
    embedding_model="text-embedding-3-large",
    
    # ===== TAXONOMY =====
    cluster_tier1_target_k=5,     # Number of broad topics
    cluster_tier2_target_k=3,     # Subtopics per topic
    cluster_tier3_target_k=2,     # Fine-grained topics
    taxonomy_approval_required=True,
    
    # ===== COST CONTROLS =====
    max_cost_per_run=5.0,         # $5 budget limit
    enable_cost_tracking=True,
    batch_api_calls=True,         # 50% discount
    enable_result_caching=True,
)

# Display configuration
print(config.display_config())

### Expected Output:
```
============================================================
RAG PDF System Configuration
============================================================
Drive folder: Research_PDFs
Max papers per run: 20
Max pages per paper: unlimited
Max chunks per paper: 50

Models:
  Summary: gpt-5-mini
  Taxonomy: gpt-5-mini
  Classification: gpt-5-mini
  Embedding: text-embedding-3-large

Reasoning Effort:
  Summary: medium
  Taxonomy: high
  Classification: medium

Clustering:
  Tier 1 target k: 5
  Tier 2 target k: 3
  Tier 3 target k: 2

Budget & Cost Controls:
  Max cost per run: $5.0
  Cost tracking: True
  Batch API calls: True
============================================================
```

## Step 6: Run the Pipeline

Execute the full RAG pipeline on your corpus.

In [ ]:
# Run the full pipeline
print("Starting pipeline execution...")
print("This may take several minutes depending on corpus size.\n")

try:
    final_state = run_full_pipeline(config)
    print("\n✅ Pipeline completed successfully!")
except Exception as e:
    print(f"\n❌ Pipeline error: {e}")
    raise

In [ ]:
# Display workflow state
print(display_workflow_state(final_state))

### Expected Output:
```
============================================================
WORKFLOW STATE
============================================================
Current Phase: export
Total Papers: 20
Pending: 0
Completed: 19
Failed: 1
Total Chunks: 847
Has Taxonomy: True
Taxonomy Approved: True
============================================================
```

In [ ]:
# View cost summary
print_cost_summary(final_state)

## Step 7: Explore the Taxonomy

View the automatically generated topic hierarchy.

In [ ]:
# Display taxonomy
if final_state.get('topic_hierarchy'):
    hierarchy = final_state['topic_hierarchy']
    print(f"Taxonomy Version: {hierarchy.taxonomy_version}")
    print(f"Total Papers: {hierarchy.total_papers}\n")
    
    for t1 in hierarchy.tier1:
        print(f"\n📁 {t1.label} ({len(t1.paper_ids)} papers)")
        print(f"   {t1.description}")
        
        for t2 in hierarchy.get_tier2_topics(t1.id):
            print(f"   └─ 📂 {t2.label} ({len(t2.paper_ids)})")
else:
    print("No taxonomy generated yet.")

### Expected Output:
```
Taxonomy Version: v1.0_20251125
Total Papers: 20

📁 Large Language Models (8 papers)
   Research on large-scale language models and their applications
   └─ 📂 Model Architecture (3)
   └─ 📂 Training Methods (3)
   └─ 📂 Applications (2)

📁 Computer Vision (6 papers)
   Visual recognition, generation, and understanding
   └─ 📂 Vision Transformers (4)
   └─ 📂 Object Detection (2)

📁 Reinforcement Learning (4 papers)
   Learning through interaction and reward optimization
   └─ 📂 Policy Optimization (2)
   └─ 📂 Multi-Agent Systems (2)

[...]
```

## Step 8: Query Your Corpus

Use RAG to ask questions about your research papers.

In [ ]:
# Initialize query engine
engine = RAGQueryEngine(final_state)
print("✅ Query engine initialized")

In [ ]:
# Example query 1: Overview question
result = engine.query(
    "What are the main research themes in this corpus?",
    top_k=5
)

print("Query: What are the main research themes in this corpus?\n")
print("Answer:")
print(result['answer'])
print("\nSources:")
for source in result['sources'][:3]:
    print(f"  - {source['title']}")

In [ ]:
# Example query 2: Specific question
result = engine.query(
    "What methods are used to improve model efficiency?",
    top_k=5
)

print("Query: What methods are used to improve model efficiency?\n")
print("Answer:")
print(result['answer'])

In [ ]:
# Interactive query with detailed output
result = interactive_query(
    final_state,
    query="What are the key findings and contributions across papers?",
    top_k=5,
    rerank=True
)

## Step 9: Search and Filter Papers

Use search utilities to find specific papers.

In [ ]:
from corpus_utilities import (
    search_by_title,
    search_by_author,
    search_by_topic,
    get_topic_distribution
)

# Search by title keyword
papers = search_by_title(final_state, "transformer")
print(f"Papers about 'transformer': {len(papers)}")
for paper in papers[:3]:
    print(f"  - {paper.title}")

print()

In [ ]:
# Get topic distribution
distribution = get_topic_distribution(final_state)
print("\nTopic Distribution:")
for topic, count in distribution.items():
    print(f"  {topic}: {count} papers")

## Step 10: Export Results

Export your processed corpus to various formats.

In [ ]:
from export_manager import export_final_data
from corpus_utilities import generate_bibtex_entries, create_reading_list

# Export all data
export_paths = export_final_data(final_state, "./exports")

print("Exported files:")
for key, path in export_paths.items():
    print(f"  {key}: {path}")

In [ ]:
# Generate BibTeX
bibtex = generate_bibtex_entries(final_state)
with open("./exports/bibliography.bib", "w") as f:
    f.write(bibtex)
print("✅ BibTeX exported to ./exports/bibliography.bib")

In [ ]:
# Create reading list
reading_list = create_reading_list(
    final_state,
    paper_ids=list(final_state['papers'].keys())[:10],
    output_path="./exports/reading_list.md",
    title="Research Reading List"
)
print("✅ Reading list exported to ./exports/reading_list.md")

## Step 11: Save Checkpoint

Save your work to resume later.

In [ ]:
# Save checkpoint
checkpoint_path = save_checkpoint(final_state)
print(f"✅ Checkpoint saved to: {checkpoint_path}")
print("\nTo resume later, use:")
print(f'  state = load_checkpoint("{checkpoint_path.split("/")[-1].replace(".pkl", "")}")')

## Step 12: Quality Control Report

Generate a quality control report for your corpus.

In [ ]:
# Generate QC report
report = generate_qc_report(final_state, "./exports/qc_report.md")

# Display summary
dashboard = create_qc_dashboard(final_state)
print(dashboard.get_summary())

---

## 🎉 Done!

You've successfully:
- ✅ Processed your PDF corpus
- ✅ Generated a topic taxonomy
- ✅ Classified papers into topics
- ✅ Enabled RAG queries
- ✅ Exported results

### Next Steps

1. **Add more papers**: Increase `max_papers_per_run` and rerun
2. **Refine taxonomy**: Adjust `cluster_tier1_target_k` for more/fewer topics
3. **Deep analysis**: Enable `enable_deep_analysis_pass=True` for detailed summaries
4. **Explore queries**: Try different questions to explore your corpus

### Resources

- [USER_GUIDE.md](USER_GUIDE.md) - Complete user guide
- [EXAMPLES.md](EXAMPLES.md) - More configuration and query examples
- [FINAL_NOTEBOOK_ACTION_PLAN.md](FINAL_NOTEBOOK_ACTION_PLAN.md) - Full implementation details

---

**Version:** 1.0.0 | **Last Updated:** 2025-11-25